# ETL VRA/ANAC — 2021

**ETL:** Arthur. **ELT:** Murilo Acioli. **Contrato:** v1.1.

A implementação está em `scripts/etl_vra_anac_2021.py`. Este notebook configura a
execução e apresenta as evidências. O Python transforma os dados antes da carga.
A entrada e `Codeshare` são preservados; as duplicatas usam os 20 campos brutos.

Use o kernel da `.venv` e execute de cima para baixo. Se alterar o script, reinicie
o kernel antes de executar novamente. Dependências: `requirements.txt`.

## 1. Configurar

Entrada: 12 CSVs em `dados/brutos/`. Saída: `dados/processados/2021/etl/`.
A carga no PostgreSQL começa desativada. Para habilitar, configure o `.env` e as
opções abaixo. O banco precisa existir; o schema usado será `etl_2021`.

Use `DATABASE_URL` PostgreSQL ou `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` e
`PGPASSWORD` no `.env` (local) ou no ambiente/Secrets do Colab. Nunca inclua
credenciais no notebook. `SUBSTITUIR_CARGA_POSTGRES=True` permite substituir as
dez tabelas deste pipeline em uma transação; use primeiro um banco de teste.
O status `concluida` exige confirmação da transação. Contagens e chaves de origem
são verificadas no banco. Consulte `status_carga_postgresql.json`.


**No Colab:** abra o notebook a partir do GitHub. No painel de arquivos, faça o
clone do repositório e use `%cd /content/Projeto-de-integracao-BD` em uma célula
temporária. Instale `requirements.txt` com `%pip install -r requirements.txt` e
reinicie a sessão se o Colab solicitar. Disponibilize os 12 CSVs em `dados/brutos/`
(por upload ou Google Drive montado); se preferir outro diretório, ajuste
`PASTA_DADOS`. Execute as quatro células abaixo, em ordem. Baixe a pasta de saída
antes de encerrar a sessão. PostgreSQL exige endereço acessível pelo Colab;
`localhost` do seu computador não é o `localhost` da sessão remota.
A reprodução no Colab ainda precisa ser executada e registrada pelo grupo.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
while not (RAIZ / "scripts" / "etl_vra_anac_2021.py").is_file():
    if RAIZ == RAIZ.parent:
        raise FileNotFoundError("Abra o notebook dentro do repositório do projeto.")
    RAIZ = RAIZ.parent

if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))
from etl_vra_anac_2021 import run_etl

PASTA_DADOS = RAIZ / "dados" / "brutos"
PASTA_SAIDA = RAIZ / "dados" / "processados" / "2021" / "etl"
EXECUTAR_CARGA_POSTGRES = False
SUBSTITUIR_CARGA_POSTGRES = False

print("Python:", sys.executable)
print("Entrada:", PASTA_DADOS)
print("Saída:", PASTA_SAIDA)

Python: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/.venv/bin/python
Entrada: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/dados/brutos
Saída: /home/arthurzanardi/Documentos/BD/Projeto-de-integracao-BD/dados/processados/2021/etl


## 2. Executar o ETL

O script valida os arquivos, deduplica, padroniza textos e tipos, separa rejeições,
calcula atrasos, audita conflitos e exporta as evidências. `Referência` aceita os
dois formatos de 2021. Opcionais inválidos não rejeitam o voo por si só.

Os originais são copiados byte a byte para `vra_bruta/`, com SHA-256.
Rastreabilidade inclui `data_carga` UTC, `pipeline` e `versao_contrato`.


In [2]:
resultado = run_etl(
    input_dir=PASTA_DADOS,
    output_dir=PASTA_SAIDA,
    load_database=EXECUTAR_CARGA_POSTGRES,
    replace=SUBSTITUIR_CARGA_POSTGRES,
    schema="etl_2021",
)
print("PostgreSQL:", resultado["carga_postgresql"]["status"])

[ETL 2021] Extraindo e validando os 12 arquivos.
Total de registros com Codeshare preenchido: 112479
Total de registros: 632,148
Total de colunas, incluindo metadados: 28
Conferência estrutural e de contagens concluída.
Registros envolvidos em repetições: 0
Grupos repetidos: 0
Ocorrências excedentes: 0
Grupos com variação em Codeshare: 0
Nenhum grupo repetido apresentou variação em Codeshare.
Registros brutos: 632148
Excedentes separados: 0
Registros após deduplicação: 632148
Registros na camada padronizada: 632148
Excedentes potenciais somente após TRIM/UPPER: 0
Essas repetições são auditadas; não são removidas automaticamente.
Registros rejeitados: 25511
Registros elegíveis: 606637
Ocorrências na auditoria de anomalias: 0
Partidas com diferença absoluta >24h: 371
Chegadas com diferença absoluta >24h: 373
Códigos com descrições conflitantes: 9
Excedentes potenciais após conversão: 0
Conciliações e verificações internas concluídas.
Comparação dos hashes e dos valores por registro com o

## 3. Conferir os resultados

O script verifica: **brutos = excedentes + rejeitados + tratados**, por mês e no ano.
A comparação abaixo usa os totais informados por Murilo. A conferência posterior
dos inventários confirmou os 12 hashes, tamanhos e contagens; a evidência está em
`resultados/validacoes/comparacao_etl_elt_2021.json`. A igualdade dos valores por
registro ainda depende da tabela tratada completa do ELT.


In [3]:
display(pd.DataFrame([resultado["resumo"]]).T.rename(columns={0: "resultado"}))
display(pd.read_csv(PASTA_SAIDA / "mensal.csv"))
display(pd.read_csv(PASTA_SAIDA / "comparativo_totais_elt_informados.csv"))

,resultado
ano_origem,2021
bruta,632148
duplicatas_excedentes,0
intersecao_excedentes_duplicados_invalidos,0
rejeicoes,25511
tratada,606637
grupos_duplicados,0
registros_envolvidos_duplicatas,0
duplicatas_apos_trim_upper,0
duplicatas_apos_padronizacao,0


,arquivo_origem,bruta,duplicatas,rejeicoes,tratada,conciliada
0,VRA_2021_01.csv,61920,0,3505,58415,True
1,VRA_2021_02.csv,46078,0,2486,43592,True
2,VRA_2021_03.csv,41701,0,2326,39375,True
3,VRA_2021_04.csv,30224,0,1250,28974,True
4,VRA_2021_05.csv,37341,0,1739,35602,True
5,VRA_2021_06.csv,42512,0,1680,40832,True
6,VRA_2021_07.csv,55555,0,2335,53220,True
7,VRA_2021_08.csv,56367,0,1517,54850,True
8,VRA_2021_09.csv,58362,0,1881,56481,True
9,VRA_2021_10.csv,61288,0,1860,59428,True


,indicador,etl_medido,elt_informado,totais_iguais
0,bruta,632148,632148,True
1,duplicatas_excedentes,0,0,True
2,rejeicoes,25511,25511,True
3,tratada,606637,606637,True


## 4. Revisar as auditorias

Abaixo: motivos das rejeições, conflitos de descrições e variação de esquema.
Os registros completos estão nos CSVs da pasta de saída. Diferenças acima de 24h
são sinalizadas e preservadas. A explicação da execução está em `LEIA_ME.md`.

Rejeições usam campos canônicos, originais `bruto__*` e `motivo_rejeicao` (lista JSON).
Anomalias identificam o campo; `partida_ano_diverge_flag` preserva partidas fora de 2021.
Possíveis duplicatas após TRIM/UPPER são somente auditadas. O log por arquivo está
em `etl_log_arquivos.csv`; excedentes inválidos não são contados novamente como rejeições.


In [4]:
for nome in ["motivos_rejeicao.csv", "vra_conflitos_codigo.csv", "auditoria_esquema.csv"]:
    print(nome)
    display(pd.read_csv(PASTA_SAIDA / nome))
print("Log por arquivo")
display(pd.read_csv(PASTA_SAIDA / "etl_log_arquivos.csv"))


motivos_rejeicao.csv
vra_conflitos_codigo.csv
auditoria_esquema.csv
Log por arquivo


,motivo,quantidade
0,CHEGADA_PREVISTA_AUSENTE,25511
1,PARTIDA_PREVISTA_AUSENTE,25511


,tipo,codigo,descricao,ocorrencias
0,aeroporto,KAEX,"ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...",5
1,aeroporto,KAEX,"ALEXANDRIA INTERNATIONAL AIRPORT - ALEXANDRIA,...",21
2,aeroporto,SWHP,OLHOS D´ÁGUA - ÁGUA BOA - MG - BRASIL,109
3,aeroporto,SWHP,OLHOS D´ÁGUA - ÁGUA BOA - MT - BRASIL,310
4,aeroporto,SYEC,EUGENE F. CORREIRA INTERNATIONAL - OGLE - - G...,14
5,aeroporto,SYEC,EUGENE F. CORREIRA INTERNATIONAL - OGLE - GUIANA,6
6,companhia,CQB,APUÍ TÁXI AÉREO LTDA.,102
7,companhia,CQB,APUÍ TÁXI AÉREO S/A,27
8,companhia,DTA,TAAG LINHAS AÉREAS DA ANGOLA (ANGOLA AIRLINES),26
9,companhia,DTA,TAAG LINHAS AÉREAS DE ANGOLA (ANGOLA AIRLINES),33


,arquivo,registros,codeshare_preenchido,exemplos,ano_origem,tipo_ocorrencia,decisao
0,VRA_2021_01.csv,61920,23061,"[""ETD/4375"", ""ETD/4399"", ""GLO/5007, KLM/2411"",...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
1,VRA_2021_02.csv,46078,18763,"[""ETD/4375"", ""ETD/4399"", ""GLO/5007, KLM/2411"",...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
2,VRA_2021_03.csv,41701,19217,"[""ETD/4375"", ""ETD/4399"", ""GLO/5007, KLM/2411"",...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
3,VRA_2021_07.csv,55555,15357,"[""ETD/4375"", ""ETD/4399"", ""GLO/5007, KLM/2411"",...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
4,VRA_2021_09.csv,58362,15577,"[""ETD/4375"", ""ETD/4399"", ""GLO/5004, KLM/2412"",...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
5,VRA_2021_11.csv,66649,4219,"[""JBU/4436"", ""JBU/4437"", ""JBU/4436 início: 27/...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...
6,VRA_2021_12.csv,74151,16285,"[""JBU/4436"", ""JBU/4437"", ""JBU/4451"", ""AZU/5340...",2021,VARIACAO_ESQUEMA_CODESHARE,Preservar na base bruta e auditar; manter os 2...


,arquivo_origem,mes,bytes,sha256,colunas_originais,registros_csv,registros_pandas,duplicatas,rejeicoes,tratada,conciliada,ocorrencias_anomalias,excedentes_invalidos,data_carga,pipeline,versao_contrato,sha256_copia,copia_bruta_verificada
0,VRA_2021_01.csv,1,18281868,6ee26c0fef40bf502f0557abd4d321a81ed0e9abb0c101...,21,61920,61920,0,3505,58415,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,6ee26c0fef40bf502f0557abd4d321a81ed0e9abb0c101...,True
1,VRA_2021_02.csv,2,13667479,693a2bacdad0a3cedaa6128c3cc10bb0691ccf23958c5e...,21,46078,46078,0,2486,43592,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,693a2bacdad0a3cedaa6128c3cc10bb0691ccf23958c5e...,True
2,VRA_2021_03.csv,3,12444242,b590ba0323cbd37ef50f9272cf067ffe4176607516222b...,21,41701,41701,0,2326,39375,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,b590ba0323cbd37ef50f9272cf067ffe4176607516222b...,True
3,VRA_2021_04.csv,4,8455519,a55b75cceb5e90fa8f8d8ed0b7910b8dfcb2c45f9dddde...,20,30224,30224,0,1250,28974,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,a55b75cceb5e90fa8f8d8ed0b7910b8dfcb2c45f9dddde...,True
4,VRA_2021_05.csv,5,10464126,49907e9546d02fd3cbbdcc5dc62bb8d64130ce91c03f05...,20,37341,37341,0,1739,35602,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,49907e9546d02fd3cbbdcc5dc62bb8d64130ce91c03f05...,True
5,VRA_2021_06.csv,6,11928409,babaf8882b29fa3b89a83df226af6385519fbe222ae335...,20,42512,42512,0,1680,40832,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,babaf8882b29fa3b89a83df226af6385519fbe222ae335...,True
6,VRA_2021_07.csv,7,16716773,fa1a34e334ed3faa5ad5d9bc1e7c39364744b68a4b8fd8...,21,55555,55555,0,2335,53220,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,fa1a34e334ed3faa5ad5d9bc1e7c39364744b68a4b8fd8...,True
7,VRA_2021_08.csv,8,15837098,4dd71d230cbe058fde0a645fac3897fbd44a33cd61326a...,20,56367,56367,0,1517,54850,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,4dd71d230cbe058fde0a645fac3897fbd44a33cd61326a...,True
8,VRA_2021_09.csv,9,17466130,f25fa7df2051735f0b09046f6242f34e9abb435dbf3dfe...,21,58362,58362,0,1881,56481,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,f25fa7df2051735f0b09046f6242f34e9abb435dbf3dfe...,True
9,VRA_2021_10.csv,10,17231843,eb18165d34365084e57c3f6a7f6cbbad9428f06c20f906...,20,61288,61288,0,1860,59428,True,0,0,2026-11-17T12:38:09.136779+00:00,ETL,1.1,eb18165d34365084e57c3f6a7f6cbbad9428f06c20f906...,True


## Pendências para integração

- Obter a tabela tratada **completa** do ELT; o inventário com SHA-256 já foi conferido.
- Executar `scripts/comparar_etl_elt_2021.py` conforme o README e revisar divergências.
- Reproduzir o notebook no Colab e registrar o resultado; a execução local não comprova essa etapa.
- Resolver coletivamente os conflitos de descrição antes de publicar dimensões e Fato.

A carga PostgreSQL é opcional. Os testes de integração usam um banco temporário
isolado; o status da execução deste notebook está em `status_carga_postgresql.json`.
Os dados completos ficam em `dados/processados/`, fora do Git; as evidências
pequenas ficam em `resultados/`. Credenciais permanecem no `.env` ou no ambiente.
